# Wafer Yield Intelligence — End-to-End Walkthrough

This notebook demonstrates the complete pipeline:

1. **Synthetic wafer map generation** — 9 defect pattern classes on a 52×52 binary grid
2. **59-feature spatial engineering** — density (13) · Radon transform (40) · geometric shape (6)
3. **Model training** — VotingEnsemble (pattern) + VotingClassifier (retest) with 5-fold CV
4. **Evaluation** — confusion matrices, per-class accuracy, GradientBoosting feature importance
5. **Inference on unseen wafers** — latency benchmark (ms/wafer)

> **To reproduce:** `pip install -r ../requirements.txt` then run all cells top-to-bottom.  
> Pre-trained models are in `../models/`; to retrain from scratch run `python ../scripts/train_pipeline.py`.

In [ ]:
import sys, json, math, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.ndimage import rotate as nd_rotate
from scipy.interpolate import CubicSpline
from skimage.measure import regionprops, label as sk_label
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

warnings.filterwarnings("ignore")

ROOT       = Path("..").resolve()
MODELS_DIR = ROOT / "models"
ART_DIR    = ROOT / "artifacts"
FIG_DIR    = ART_DIR / "figures"

PATTERN_CLASSES = ["none","Center","Donut","Edge-Local","Edge-Ring","Local","Random","Scratch","Near-full"]
WAFER_SIZE = 52
RNG = np.random.default_rng(42)
palette = sns.color_palette("muted", 9)
print("Setup complete.")

## 1 — Synthetic Wafer Map Generation

Nine defect pattern classes generated on a binary 52×52 circular wafer grid.  
Each class follows a geometric rule (ring, line, cluster, etc.) plus ~1-2% random noise.

In [ ]:
def _wafer_mask(n=WAFER_SIZE):
    c = n / 2
    Y, X = np.ogrid[:n, :n]
    return ((X-c)**2 + (Y-c)**2) <= (c-1)**2

def _make_pattern(name, n=WAFER_SIZE, rng=RNG):
    mask = _wafer_mask(n); w = np.zeros((n,n), dtype=np.uint8); c = n/2
    if name == "none":   w[mask & (rng.random((n,n)) < 0.02)] = 1
    elif name == "Center":
        Y,X = np.ogrid[:n,:n]
        w[mask & (((X-c)**2+(Y-c)**2) < (n*0.2)**2)] = 1
        w[mask & (rng.random((n,n)) < 0.02)] = 1
    elif name == "Donut":
        Y,X = np.ogrid[:n,:n]; d=(X-c)**2+(Y-c)**2
        w[mask & (d>(n*0.15)**2) & (d<(n*0.3)**2)] = 1
        w[mask & (rng.random((n,n)) < 0.02)] = 1
    elif name == "Edge-Local":
        Y,X = np.ogrid[:n,:n]; a=rng.uniform(0,2*math.pi)
        ex,ey = c+(c-3)*np.cos(a), c+(c-3)*np.sin(a)
        w[mask & (((X-ex)**2+(Y-ey)**2)<(n*0.15)**2)] = 1
        w[mask & (rng.random((n,n)) < 0.01)] = 1
    elif name == "Edge-Ring":
        Y,X = np.ogrid[:n,:n]
        w[mask & (np.sqrt((X-c)**2+(Y-c)**2) > (c-5))] = 1
        w[mask & (rng.random((n,n)) < 0.01)] = 1
    elif name == "Local":
        Y,X = np.ogrid[:n,:n]
        lx,ly = rng.integers(n//4,3*n//4), rng.integers(n//4,3*n//4)
        w[mask & (((X-lx)**2+(Y-ly)**2)<(n*0.1)**2)] = 1
        w[mask & (rng.random((n,n)) < 0.02)] = 1
    elif name == "Random": w[mask & (rng.random((n,n)) < rng.uniform(0.10,0.25))] = 1
    elif name == "Scratch":
        Y,X = np.meshgrid(np.arange(n),np.arange(n),indexing="ij")
        ang = rng.uniform(-0.5,0.5)
        w[mask & (np.abs(Y-(c+ang*(X-c)))<2)] = 1
        w[mask & (rng.random((n,n)) < 0.01)] = 1
    elif name == "Near-full": w[mask & (rng.random((n,n)) < rng.uniform(0.60,0.85))] = 1
    return w

fig, axes = plt.subplots(3, 3, figsize=(9,9))
for idx, cls in enumerate(PATTERN_CLASSES):
    ax = axes[idx//3, idx%3]
    wmap = _make_pattern(cls, rng=np.random.default_rng(idx))
    ax.imshow(wmap, cmap="RdYlGn_r", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"{cls}  ({100*wmap.sum()/_wafer_mask().sum():.1f}% defect)", fontsize=10, fontweight="bold")
    ax.axis("off")
plt.suptitle("Synthetic Wafer Defect Patterns  (52 × 52 grid, binary)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR/"wafer_patterns_nb.png", dpi=150, bbox_inches="tight"); plt.show()

## 2 — 59-Feature Spatial Engineering

Three complementary groups — density regions, Radon-transform projections, and geometric shape descriptors.  
**Why Radon?** A Scratch produces a sharp sinogram spike at the perpendicular angle; a Donut is flat across all angles — this is the key discriminator.

In [ ]:
def density_features(wmap, mask, n=13):
    sz = wmap.shape[0]; c = sz/2
    Y,X = np.ogrid[:sz,:sz]; dist = np.sqrt((X-c)**2+(Y-c)**2)
    feats = []
    for frac in [0.2,0.4,0.6,0.8,1.0]:
        r = mask & (dist < c*frac)
        feats.append(wmap[r].sum()/r.sum() if r.sum()>0 else 0.0)
    ang = np.arctan2(Y-c, X-c)
    for i in range(8):
        lo=-math.pi+i*math.pi/4; r=mask&(ang>=lo)&(ang<lo+math.pi/4)
        feats.append(wmap[r].sum()/r.sum() if r.sum()>0 else 0.0)
    return feats[:n]

def radon_features(wmap, n_proj=20):
    means,stds = [],[]
    for a in np.linspace(0,180,n_proj,endpoint=False):
        rot = nd_rotate(wmap.astype(float),a,reshape=False,order=1)
        proj = rot.sum(axis=0)
        if len(proj)>3:
            cs=CubicSpline(np.arange(len(proj)),proj)
            v=cs(np.linspace(0,len(proj)-1,50))
            means.append(float(v.mean())); stds.append(float(v.std()))
        else: means.append(0.); stds.append(0.)
    return means+stds

def geom_features(wmap):
    props = regionprops(sk_label(wmap))
    if not props: return [0.]*6
    p = max(props, key=lambda x:x.area)
    return [float(p.area), float(p.perimeter or 0),
            float(p.axis_major_length), float(p.axis_minor_length),
            float(p.eccentricity), float(p.solidity)]

def extract_59(wmap):
    mask = _wafer_mask(wmap.shape[0])
    return density_features(wmap,mask) + radon_features(wmap) + geom_features(wmap)

assert len(extract_59(_make_pattern("Scratch"))) == 59
print("Feature vector: 13 density + 40 Radon + 6 geometric = 59 ✓")

# Compare Scratch vs Donut feature signatures
fig, axes = plt.subplots(2, 3, figsize=(13,6))
COLS = {"Scratch":"#E74C3C","Donut":"#2980B9"}
for row,(cls,c) in enumerate(COLS.items()):
    wmap = _make_pattern(cls, rng=np.random.default_rng(1))
    fv = extract_59(wmap)
    axes[row,0].imshow(wmap, cmap="RdYlGn_r", vmin=0, vmax=1, interpolation="nearest")
    axes[row,0].set_title(f"{cls} — wafer map", fontsize=11, fontweight="bold"); axes[row,0].axis("off")
    axes[row,1].bar(range(13), fv[:13], color=c, alpha=0.85)
    axes[row,1].set_title("Density features (13)", fontsize=11)
    axes[row,1].set_xlabel("Region"); axes[row,1].set_ylabel("Defect fraction")
    axes[row,2].plot(fv[13:33], color=c, lw=1.8, label="Radon mean")
    axes[row,2].plot(fv[33:53], color=c, lw=1.8, ls="--", alpha=0.6, label="Radon std")
    axes[row,2].set_title("Radon features (40)", fontsize=11)
    axes[row,2].set_xlabel("Angle index (0°→180°)"); axes[row,2].legend(fontsize=9)
plt.suptitle("Feature Signatures: Scratch (spike) vs Donut (flat) in Radon space", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig(FIG_DIR/"feature_comparison_nb.png", dpi=150, bbox_inches="tight"); plt.show()

## 3 — Model Training (5-fold Stratified CV)

In [ ]:
# Generate datasets
print("Generating 900 pattern wafers …")
X_pat, y_pat = [], []
for cls in PATTERN_CLASSES:
    for _ in range(100):
        X_pat.append(extract_59(_make_pattern(cls))); y_pat.append(cls)
X_pat, y_pat = np.array(X_pat), np.array(y_pat)

print("Generating 5,000 retest records …")
X_ret = RNG.normal(0,1,(5000,28))
y_ret = (np.abs(X_ret).mean(1) + RNG.normal(0,0.3,5000) > 1.0).astype(int)
print(f"Retest: {(y_ret==0).sum()} Fail / {(y_ret==1).sum()} Pass")

Xp_tr,Xp_te,yp_tr,yp_te = train_test_split(X_pat,y_pat,test_size=0.2,stratify=y_pat,random_state=42)
Xr_tr,Xr_te,yr_tr,yr_te = train_test_split(X_ret,y_ret,test_size=0.2,stratify=y_ret,random_state=42)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

# Pattern model
p_sc = StandardScaler(); Xp_s = p_sc.fit_transform(Xp_tr)
ve = VotingClassifier([("gb",GradientBoostingClassifier(n_estimators=150,max_depth=5,learning_rate=0.1,random_state=42)),
                       ("mlp",MLPClassifier(hidden_layer_sizes=(128,64,32),max_iter=500,random_state=42))],voting="soft")
p_cv = cross_val_score(ve,Xp_s,yp_tr,cv=cv,scoring="accuracy")
ve.fit(Xp_s,yp_tr)
print(f"\nPattern VotingEnsemble  CV={p_cv.mean():.1%}±{p_cv.std():.1%}  Test={accuracy_score(yp_te,ve.predict(p_sc.transform(Xp_te))):.1%}")

# Retest model
r_sc = StandardScaler(); Xr_s = r_sc.fit_transform(Xr_tr)
vc = VotingClassifier([("knn",KNeighborsClassifier(n_neighbors=5,weights="distance")),
                       ("rf",RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)),
                       ("mlp",MLPClassifier(hidden_layer_sizes=(64,32),max_iter=300,random_state=42))],voting="soft")
r_cv = cross_val_score(vc,Xr_s,yr_tr,cv=cv,scoring="accuracy")
vc.fit(Xr_s,yr_tr)
print(f"Retest VotingClassifier CV={r_cv.mean():.1%}±{r_cv.std():.1%}  Test={accuracy_score(yr_te,vc.predict(r_sc.transform(Xr_te))):.1%}")

## 4 — Evaluation: Confusion Matrices + Feature Importance

In [ ]:
yp_pred = ve.predict(p_sc.transform(Xp_te))
yr_pred = vc.predict(r_sc.transform(Xr_te))

fig, axes = plt.subplots(1,2,figsize=(16,6))
cm_p = confusion_matrix(yp_te,yp_pred,labels=PATTERN_CLASSES)
sns.heatmap(cm_p.astype(float)/cm_p.sum(1,keepdims=True).clip(1), annot=True, fmt=".2f",
            cmap="Blues", xticklabels=PATTERN_CLASSES, yticklabels=PATTERN_CLASSES, ax=axes[0], linewidths=0.3)
axes[0].set_title(f"Pattern Recognition  Acc={accuracy_score(yp_te,yp_pred):.1%}", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Actual"); axes[0].set_xlabel("Predicted"); axes[0].tick_params(axis="x",rotation=40,labelsize=9)

cm_r = confusion_matrix(yr_te,yr_pred)
sns.heatmap(cm_r,annot=True,fmt="d",cmap="Oranges",xticklabels=["Fail","Pass"],yticklabels=["Fail","Pass"],ax=axes[1],linewidths=0.3)
axes[1].set_title(f"Retest Prediction  Acc={accuracy_score(yr_te,yr_pred):.1%}", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Actual"); axes[1].set_xlabel("Predicted")
plt.tight_layout(); plt.savefig(FIG_DIR/"confusion_matrices_nb.png",dpi=150,bbox_inches="tight"); plt.show()

# Per-class accuracy + feature importance
fig, axes = plt.subplots(1,2,figsize=(15,5))
per_cls = cm_p.diagonal()/cm_p.sum(1).clip(1)
bars = axes[0].bar(PATTERN_CLASSES, per_cls, color=palette)
axes[0].set_ylim(0,1.1); axes[0].set_ylabel("Accuracy"); axes[0].tick_params(axis="x",rotation=35)
axes[0].set_title("Per-Class Pattern Accuracy", fontsize=12, fontweight="bold")
for b,v in zip(bars,per_cls): axes[0].text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.0%}", ha="center", fontsize=9)

importances = ve.named_estimators_["gb"].feature_importances_
fnames = [f"den_{i}" for i in range(13)] + [f"rad_μ{i}" for i in range(20)] + [f"rad_σ{i}" for i in range(20)] + ["area","perim","maj","min","ecc","sol"]
top20 = np.argsort(importances)[-20:]
axes[1].barh([fnames[i] for i in top20], importances[top20], color=sns.color_palette("viridis",20))
axes[1].set_title("Top-20 Feature Importances (GradientBoosting)", fontsize=12, fontweight="bold"); axes[1].set_xlabel("Importance")
plt.tight_layout(); plt.savefig(FIG_DIR/"evaluation_nb.png",dpi=150,bbox_inches="tight"); plt.show()

## 5 — Inference on Unseen Wafers + Latency

In [ ]:
rng_u = np.random.default_rng(777)  # different seed — no overlap with training
X_u, y_u = [], []
for cls in PATTERN_CLASSES:
    for _ in range(10):
        X_u.append(extract_59(_make_pattern(cls, rng=rng_u))); y_u.append(cls)
X_u, y_u = np.array(X_u), np.array(y_u)

t0 = time.perf_counter()
pat_u = ve.predict(p_sc.transform(X_u))
pat_ms = (time.perf_counter()-t0)*1000

t1 = time.perf_counter()
ret_u = vc.predict(r_sc.transform(X_u[:,:28]))
ret_ms = (time.perf_counter()-t1)*1000

acc_u = accuracy_score(y_u, pat_u)
print(f"Unseen holdout  pattern accuracy : {acc_u:.1%}  ({int(acc_u*90)}/90)")
print(f"Retest pass rate                 : {(ret_u==1).mean():.1%}")
print(f"\nInference latency:")
print(f"  Pattern VotingEnsemble : {pat_ms:.1f} ms total | {pat_ms/90:.3f} ms/wafer")
print(f"  Retest  VotingClassifier: {ret_ms:.1f} ms total | {ret_ms/90:.2f} ms/wafer")

summary = pd.DataFrame({
    "Model":["Pattern VotingEnsemble","Retest VotingClassifier"],
    "CV Accuracy":[f"{p_cv.mean():.1%} ±{p_cv.std():.1%}", f"{r_cv.mean():.1%} ±{r_cv.std():.1%}"],
    "Test Accuracy":[f"{accuracy_score(yp_te,yp_pred):.1%}", f"{accuracy_score(yr_te,yr_pred):.1%}"],
    "Unseen (90 wafers)":[f"{acc_u:.1%}","—"],
    "ms / wafer":[f"{pat_ms/90:.3f}", f"{ret_ms/90:.1f}"],
    "Features":["59 spatial","28 yield/offset"],
}).set_index("Model")
display(summary)